# T2 — Inventario y limpieza del flatfile NGA-West2

**Proyecto:** Predicción de la Aceleración Máxima del Suelo (PGA) con Machine Learning
**Curso:** Aplicaciones de IA en Estructuras — UPC
**Grupo 01:** Derick De La Vega, Diego Marin, Jorge Zelaya

**Objetivo de este notebook:** verificar el contenido real del dataset descrito en el T1,
aplicar la limpieza básica y cuantificar las fuentes de incertidumbre anticipadas en el
análisis PCS (sección 7.3 del informe T1).

**Fuente de datos:** `Updated_NGA_West2_Flatfile_RotD50_d005_public_version.xlsx`
(PEER, versión actualizada del 17/01/2015, 5% de amortiguamiento, componente RotD50).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Semilla fija: requisito de computabilidad (PCS)
SEED = 42
np.random.seed(SEED)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

print("pandas:", pd.__version__)
print("numpy :", np.__version__)

## 1. Carga de los datos

El flatfile tiene 274 columnas. Cargamos únicamente las que corresponden a las variables
definidas en la sección 6.3 del informe T1, más dos columnas adicionales útiles para
evaluar la calidad de los datos.

In [ ]:
RUTA = "../data/Updated_NGA_West2_Flatfile_RotD50_d005_public_version.xlsx"

COLUMNAS = {
    "Record Sequence Number": "RSN",                    # identificador del registro
    "EQID": "EQID",                                     # identificador del sismo (agrupación)
    "Earthquake Name": "sismo",
    "YEAR": "anio",
    "Earthquake Magnitude": "Mw",                       # fuente: magnitud de momento
    "Mechanism Based on Rake Angle": "mecanismo",       # fuente: tipo de falla
    "Hypocenter Depth (km)": "prof_hipo",               # fuente: profundidad
    "ClstD (km)": "Rrup",                               # trayectoria: distancia a la ruptura
    "Joyner-Boore Dist. (km)": "Rjb",                   # trayectoria: alternativa (judgment call)
    "Vs30 (m/s) selected for analysis": "Vs30",         # sitio
    "Measured/Inferred Class": "vs30_clase",            # sitio: medido vs inferido
    "Lowest Usable Freq - Ave. Component (Hz)": "f_min",# calidad del registro
    "PGA (g)": "PGA",                                   # variable dependiente
}

df = pd.read_excel(RUTA, usecols=list(COLUMNAS.keys()))
df = df.rename(columns=COLUMNAS)

print(f"Filas   : {len(df):,}")
print(f"Columnas: {df.shape[1]}")
df.head()

## 2. Inventario inicial

Verificamos las cifras que en el informe T1 quedaron pendientes de confirmación.

In [ ]:
print(f"Registros totales : {len(df):,}")
print(f"Sismos únicos     : {df['EQID'].nunique():,}")
print(f"Años cubiertos    : ver celda siguiente (la columna anio también usa el código -999)")
print()
print("Registros por sismo:")
print(df.groupby("EQID").size().describe().round(1))

## 3. Tratamiento de valores faltantes

Como se anticipó en la sección 6.4 del informe, el flatfile codifica los datos faltantes
con el valor `-999` en lugar de dejarlos vacíos. Si no se convierten a nulo, contaminan
cualquier estadística y cualquier modelo.

In [ ]:
num = ["Mw", "prof_hipo", "Rrup", "Rjb", "Vs30", "f_min", "PGA", "mecanismo", "anio"]

print("Conteo de códigos -999 por variable:")
for col in num:
    n = (df[col] == -999).sum()
    print(f"  {col:12s}: {n:5d}  ({n/len(df)*100:.2f} %)")

# Conversión a nulo
df[num] = df[num].replace(-999, np.nan)

print("\nNulos tras la conversión:")
print(df[num].isna().sum())

In [ ]:
print(f"Registros con PGA válida: {df['PGA'].notna().sum():,}")
print("\nValor de referencia publicado por Ancheta et al. (2014): 21,336 registros")
print("de tres componentes en la base NGA-West2.")

## 4. Estadísticas descriptivas

Rango real de las variables, para contrastar con lo declarado en el informe T1.

In [ ]:
df[["Mw", "Rrup", "Rjb", "Vs30", "prof_hipo", "PGA"]].describe().T.round(3)

## 5. Codificación del mecanismo de falla

El flatfile codifica el mecanismo numéricamente según el ángulo de rake. La correspondencia
está documentada en el reporte PEER 2013/03 (Ancheta et al., 2014), **que debe consultarse
para confirmar el significado de cada código antes de usarlo en el modelo**.

> Nota metodológica: para Random Forest y XGBoost la codificación numérica es válida.
> Para la regresión lineal y la red neuronal debe aplicarse *one-hot encoding*, porque
> de lo contrario el modelo asume un orden entre categorías que no existe.

In [ ]:
print("Distribución del mecanismo de falla:")
print(df["mecanismo"].value_counts(dropna=False).sort_index())

## 6. Calidad del Vs30: medido vs. inferido

La columna `Measured/Inferred Class` no es binaria: contiene códigos compuestos que
describen el método de estimación. Esta es una de las fuentes de incertidumbre
identificadas en la sección 7.3 del informe T1 y debe documentarse antes de decidir
si se filtran los valores inferidos.

In [ ]:
clases = df["vs30_clase"].value_counts(dropna=False)
print(f"Número de categorías distintas: {len(clases)}")
print()
print(clases.head(10))
print()
print("DECISIÓN PENDIENTE DEL GRUPO: consultar la documentación del PEER")
print("(NGA_West2_SiteDatabaseDocumentation_V032.xlsx) para interpretar estos")
print("códigos y decidir si se conservan todos los registros o solo los medidos.")

## 7. Dataset de trabajo

Conservamos únicamente los registros con información completa en las variables del modelo.

In [ ]:
esenciales = ["PGA", "Mw", "Rrup", "Vs30", "mecanismo", "prof_hipo", "EQID"]
datos = df.dropna(subset=esenciales).copy()

# Transformación logarítmica de la variable dependiente (sección 6.3 del T1):
# la PGA varía en varios órdenes de magnitud.
datos["lnPGA"] = np.log(datos["PGA"])

print(f"Registros antes  : {len(df):,}")
print(f"Registros después: {len(datos):,}  ({len(datos)/len(df)*100:.1f} %)")
print(f"Sismos           : {datos['EQID'].nunique():,}")

## 8. Distribuciones

Primera visualización de las variables. La PGA se muestra en escala logarítmica porque
en escala natural su distribución es fuertemente asimétrica.

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(15, 8))

sns.histplot(datos["Mw"], bins=40, ax=ax[0, 0], color="steelblue")
ax[0, 0].set_title("Magnitud de momento (Mw)")

sns.histplot(datos["Rrup"], bins=40, ax=ax[0, 1], color="steelblue")
ax[0, 1].set_title("Distancia a la ruptura (km)")

sns.histplot(datos["Vs30"], bins=40, ax=ax[0, 2], color="steelblue")
ax[0, 2].set_title("Vs30 (m/s)")

sns.histplot(datos["prof_hipo"], bins=40, ax=ax[1, 0], color="steelblue")
ax[1, 0].set_title("Profundidad hipocentral (km)")

sns.histplot(datos["PGA"], bins=60, ax=ax[1, 1], color="indianred")
ax[1, 1].set_title("PGA (g) — escala natural")

sns.histplot(datos["lnPGA"], bins=60, ax=ax[1, 2], color="seagreen")
ax[1, 2].set_title("ln(PGA) — escala logarítmica")

plt.tight_layout()
plt.savefig("../results/distribuciones.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Cobertura magnitud–distancia

Este gráfico muestra el **desbalance de los datos** anticipado en la sección 7.3 del T1:
hay pocos registros de sismos grandes a corta distancia, que son justamente los más
relevantes para el diseño sismorresistente.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(datos["Rrup"], datos["Mw"], s=4, alpha=0.25, color="steelblue")
ax.set_xscale("log")
ax.set_xlabel("Distancia a la ruptura, Rrup (km)")
ax.set_ylabel("Magnitud de momento, Mw")
ax.set_title("Cobertura del dataset en el espacio magnitud–distancia")

# Región crítica para diseño: sismos grandes y cercanos
ax.axhline(6.5, ls="--", c="red", lw=1)
ax.axvline(20, ls="--", c="red", lw=1)

criticos = ((datos["Mw"] >= 6.5) & (datos["Rrup"] <= 20)).sum()
ax.text(0.02, 0.02,
        f"Registros con Mw>=6.5 y Rrup<=20 km: {criticos:,} ({criticos/len(datos)*100:.1f} %)",
        transform=ax.transAxes, fontsize=9,
        bbox=dict(boxstyle="round", fc="white", ec="gray"))

plt.tight_layout()
plt.savefig("../results/cobertura_magnitud_distancia.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Verificación física: atenuación

Comprobación de que los datos reproducen el comportamiento físico esperado: la PGA debe
**disminuir con la distancia** y **aumentar con la magnitud**. Si este gráfico no muestra
ese patrón, hay un error en la selección de columnas o en la limpieza.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

bandas = [(4.0, 5.0), (5.0, 6.0), (6.0, 7.0), (7.0, 8.0)]
colores = ["#a6cee3", "#1f78b4", "#fb9a99", "#e31a1c"]

for (lo, hi), color in zip(bandas, colores):
    m = (datos["Mw"] >= lo) & (datos["Mw"] < hi)
    ax.scatter(datos.loc[m, "Rrup"], datos.loc[m, "PGA"],
               s=5, alpha=0.3, color=color, label=f"Mw {lo}–{hi}")

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Distancia a la ruptura, Rrup (km)")
ax.set_ylabel("PGA (g)")
ax.set_title("Atenuación observada de la PGA con la distancia")
ax.legend(markerscale=3)

plt.tight_layout()
plt.savefig("../results/atenuacion_observada.png", dpi=150, bbox_inches="tight")
plt.show()

## 11. Correlaciones

Matriz de correlación entre las variables del modelo. La correlación entre magnitud y
distancia es relevante porque, como se indicó en el T1, afecta la interpretación de las
técnicas de explicabilidad como SHAP.

In [ ]:
variables = ["lnPGA", "Mw", "Rrup", "Vs30", "prof_hipo"]
corr = datos[variables].corr()

fig, ax = plt.subplots(figsize=(7, 5.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            square=True, linewidths=.5, ax=ax)
ax.set_title("Matriz de correlación (Pearson)")
plt.tight_layout()
plt.savefig("../results/correlaciones.png", dpi=150, bbox_inches="tight")
plt.show()

## 12. Partición por sismo (EQID)

Decisión metodológica central declarada en la sección 7.1 del informe T1: la separación
entre entrenamiento y prueba se hace **por sismo**, no por registro.

Los registros de un mismo sismo comparten la fuente sísmica. Si se repartieran al azar,
el modelo vería en entrenamiento registros del mismo evento que debe predecir en prueba,
y la métrica resultante sobreestimaría su capacidad real de generalización.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

X = datos.drop(columns=["lnPGA", "PGA"])
y = datos["lnPGA"]
grupos = datos["EQID"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
idx_train, idx_test = next(gss.split(X, y, groups=grupos))

train = datos.iloc[idx_train]
test = datos.iloc[idx_test]

print(f"Entrenamiento: {len(train):,} registros de {train['EQID'].nunique()} sismos")
print(f"Prueba       : {len(test):,} registros de {test['EQID'].nunique()} sismos")

comunes = set(train["EQID"]) & set(test["EQID"])
print(f"\nSismos presentes en ambos conjuntos: {len(comunes)}  (debe ser 0)")

## 13. Resumen y decisiones pendientes

**Verificado en este notebook:**

- Registros totales, sismos únicos y rangos reales de cada variable.
- Códigos `-999` convertidos a nulo y cuantificados.
- Desbalance de los datos en la región crítica para diseño.
- Comportamiento de atenuación coherente con la física del problema.
- Partición por EQID sin fuga de información entre conjuntos.

**Decisiones de criterio (*judgment calls*) que el grupo debe resolver y documentar:**

1. Rango de magnitud y distancia a conservar.
2. Uso de `Rrup` o `Rjb` como medida de distancia.
3. Tratamiento de los registros con Vs30 inferido.
4. Exclusión o no de las réplicas (clase de sismo).
5. Filtro por frecuencia mínima utilizable (`f_min`).

Cada una de estas decisiones debe evaluarse en el análisis de estabilidad del T3,
verificando si las conclusiones se mantienen al modificarlas.